In [101]:
import numpy    as np
import pandas   as pd
import altair   as alt

from dataclasses    import dataclass
from typing         import Protocol, Generic, TypeVar

# `ellipses-abound`
## Definitions
An ellipse is a *plane curve* defined over two *focal points* s.t. for all points on the ellipse's curve, the sum of both distances to the two focal points is a constants. Ellipses are generalizations of the circle.

The *eccentricity* $e$ of an ellipse is a real number between $[0, 1]$ that defines its /elongation/. When $e = 1$, the ellipse elongate to a parabola, and when $e = 0$, the ellipse becomes a circle. The largest and the smallest *diameters* of an ellipse (i.e. its *width* and *height*), are denoted $2a$ and $2b$. Every ellipse also has four *extreme points*: two *vertices* at the endpoints of the *major axis* (the **width** axis), and two *co-vertices* at the endpoins of the *minor* or **height** axis.

The analytical definition for the standard ellipse, *centered at the origin*, is

$$\frac{x^{2}}{a^{2}} + \frac{y^{2}}{b^{2}} = 1$$

where $a, b \in \mathbb{R}$. The *foci* are $(\pm c, 0)$ for $c = \sqrt{a^{2} + b^{2}}$. $c$ is denoted the *linear eccentricity*, or the *distance from the center to a focus*.

The standard parametrization for ellipses centered at the origin is

$$(x, y) = (a \cdot \cos(t), b \cdot sin(t)), \, \text{for} \, 0 \leq t \leq 2 \pi$$

### The `Ellipse` class in Python
We will define now a `dataclass` with some useful methods in Python. Then, the `Plotter` class will be able to receive a `Ellipse` instance and plot it using, at first, `matplotlib`. A `Sampler` class is a parametric builder for objects.

In [102]:
#   Ellipse dataclasse definition

@dataclass
class Ellipse:
    a: float                        # semi-major axis
    b: float                        # semi-minor axis
    center: tuple[float, float]     # (x, y) coordinates of the center
    rotation: float = 0.0           # rotation angle in radians
    
    @property
    def c(self):
        """Linear eccentricity."""
        return np.sqrt(self.a**2 - self.b**2)

    @property
    def eccentricity(self):
        """Elongation of the ellipse."""
        return self.c / self.a

    @property
    def foci(self):
        """Foci of the ellipse."""
        cx, cy = self.center
        θ = self.rotation

        dx = self.c * np.cos(θ)
        dy = self.c * np.sin(θ)

        return (
            (cx - dx, cy - dy),
            (cx + dx, cy + dy)
        )

In [103]:
# `Plotter` abstract class

T: TypeVar = TypeVar("T");

class Plotter(Protocol, Generic[T]):
    """Abstract base class for plotting curve-like objects."""
    def plot(self, obj: T) -> alt.TopLevelMixin:
        """Plot the sampled points."""
        ...
        
    def coordinate_axes(self, size: int = 10):
        x_axis = pd.DataFrame({
            "x": [-size, size],
            "y": [0, 0],
        })

        y_axis = pd.DataFrame({
            "x": [0, 0],
            "y": [-size, size],
        })

        return (
            alt.Chart(x_axis)
                .mark_line(strokeDash=[6,4], color="gray")
                .encode(x="x:Q", y="y:Q")
            +
            alt.Chart(y_axis)
                .mark_line(strokeDash=[6,4], color="gray")
                .encode(x="x:Q", y="y:Q")
        );
        
    def plot_segment(self, p1, p2, color="black", dash=[6,4]):
        """Plot a line segment between two points."""
        df = pd.DataFrame({
            "x": [p1[0], p2[0]],
            "y": [p1[1], p2[1]]
        })

        return (
            alt.Chart(df)
            .mark_line(
                color=color,
                strokeDash=dash
            )
            .encode(
                x="x:Q",
                y="y:Q"
            )
        );
        
    def plot_points(self, points, color="black", size=70):
        """Plot a set of points."""
        df = pd.DataFrame(points, columns=["x", "y"])

        return (
            alt.Chart(df)
            .mark_circle(
                color=color,
                size=size
            )
            .encode(
                x="x:Q",
                y="y:Q"
            )
        )

#### Defining the `Ellipse` and `Plotter` classes
We may also want to visualize it's semi-axes and foci. Let us consider vectors $\hat{u}, \hat{v}$ such that

$$\hat{u} = (\cos \theta, \sin \theta)$$
$$\hat{v} = (-\sin \theta, \cos \theta)$$

so $\hat{u}$ moves along the *major*, width, axis and $\hat{v}$ along the height axis. In this manner, we get $C \pm a \hat{u}$ as the *major axis endpoints* and $C \pm b \hat{v}$, the *minor axis endpoints*. We may also introduce a **bouding-box** property for the ellipses.

We define the vector $\hat{u}$ and $\hat{v}$ inside the `Ellipse` class and utilize them, rather than repeating $cos \theta$ and $\sin \theta$ throughout it. For the bounding-box, we need to define the *half-widths* of an ellipse:

$$h_{x} = \sqrt{a^{2} \cos^{2} \theta + b^{2}\sin^{2} \theta}$$
$$h_{y} = \sqrt{a^{2} \sin^{2} \theta + b^{2}\cos^{2} \theta}$$

In [104]:
# Ellipse class improvements

@dataclass
class Ellipse:
    a: float                        # semi-major axis
    b: float                        # semi-minor axis
    center: tuple[float, float]     # (x, y) coordinates of the center
    rotation: float = 0.0           # rotation angle in radians
    
    @property
    def u(self) -> np.ndarray:
        """Unit vector along the major axis."""
        θ = self.rotation
        return np.array([np.cos(θ), np.sin(θ)])

    @property
    def v(self) -> np.ndarray:
        """Unit vector along the minor axis."""
        θ = self.rotation
        return np.array([-np.sin(θ), np.cos(θ)])
    
    @property
    def c(self):
        """Linear eccentricity."""
        return np.sqrt(self.a**2 - self.b**2)

    @property
    def eccentricity(self):
        """Elongation of the ellipse."""
        return self.c / self.a

    @property
    def foci(self):
        c = np.asarray(self.center)
        return (
            tuple(c - self.c * self.u),
            tuple(c + self.c * self.u)
        )
        
    @property
    def major_axis(self):
        c = np.asarray(self.center)
        return (
            tuple(c - self.a * self.u),
            tuple(c + self.a * self.u)
        )

    @property
    def minor_axis(self):
        c = np.asarray(self.center)
        return (
            tuple(c - self.b * self.v),
            tuple(c + self.b * self.v)
        )
        
    @property
    def vertices(self):
        """Vertices of the ellipse."""
        return self.major_axis;
    
    @property
    def covertices(self):
        """Co-vertices of the ellipse."""
        return self.minor_axis;        
    
    @property
    def bounding_box(self):
        cx, cy = self.center
        θ = self.rotation

        hx = np.sqrt(
            self.a**2 * np.cos(θ)**2 +
            self.b**2 * np.sin(θ)**2
        )

        hy = np.sqrt(
            self.a**2 * np.sin(θ)**2 +
            self.b**2 * np.cos(θ)**2
        )

        return (
            (cx - hx, cy - hy),
            (cx + hx, cy + hy),
        )
        
    def sample(self, num_points: int = 100) -> np.ndarray:
        """Sample points along the ellipse."""
        t = np.linspace(0, 2 * np.pi, num_points)
        x = self.a * np.cos(t)
        y = self.b * np.sin(t)

        # Apply rotation
        θ = self.rotation
        x_rot = x * np.cos(θ) - y * np.sin(θ)
        y_rot = x * np.sin(θ) + y * np.cos(θ)

        # Translate to center
        cx, cy = self.center
        x_final = x_rot + cx
        y_final = y_rot + cy

        return np.column_stack((x_final, y_final));

In [105]:
# EllipsePlotter class improvements

class EllipsePlotter(Plotter[Ellipse]):
    """Concrete implementation of the Plotter for Ellipses."""
    
    def plot_axes(self, ellipse: Ellipse, margin: float = 1.0):
        """Returns an altair CHART withe the coordinate axes of an ELLIPSE's bounding-box with a given MARGIN."""
        (xmin, ymin), (xmax, ymax) = ellipse.bounding_box

        xmin -= margin
        xmax += margin
        ymin -= margin
        ymax += margin

        x_axis = pd.DataFrame({
            "x": [xmin, xmax],
            "y": [0, 0],
        })

        y_axis = pd.DataFrame({
            "x": [0, 0],
            "y": [ymin, ymax],
        })

        return (
            alt.Chart(x_axis)
            .mark_line(strokeDash=[5,5], color="gray")
            .encode(x="x:Q", y="y:Q")
            +
            alt.Chart(y_axis)
            .mark_line(strokeDash=[5,5], color="gray")
            .encode(x="x:Q", y="y:Q")
        )
        
    def plot_major_axis(self, ellipse):
        """Returns an altair CHART with the major axis (width axis) of an ELLIPSE."""
        return self.plot_segment(
            *ellipse.major_axis,
            color="red"
        )

    def plot_minor_axis(self, ellipse):
        """Returns an altair CHART with the minor axis (height axis) of an ELLIPSE."""
        return self.plot_segment(
            *ellipse.minor_axis,
            color="green"
        )
        
    def plot_center(self, ellipse):
        """Returns an altair CHART with the center of an ELLIPSE."""
        return self.plot_points(
            [ellipse.center],
            color="black",
            size=80
        )

    def plot_vertices(self, ellipse):
        """Returns an altair CHART with the vertices of an ELLIPSE."""
        return self.plot_points(
            ellipse.vertices,
            color="red"
        )

    def plot_covertices(self, ellipse):
        """Returns an altair CHART with the co-vertices of an ELLIPSE."""
        return self.plot_points(
            ellipse.covertices,
            color="green"
        )

    def plot_foci(self, ellipse):
        """Returns an altair CHART with the foci of an ELLIPSE."""
        return self.plot_points(
            ellipse.foci,
            color="blue"
        )
    
    def plot_bounding_box(self, ellipse):
        """Returns an altair CHART with the bounding box of an ELLIPSE."""
        (xmin, ymin), (xmax, ymax) = ellipse.bounding_box

        df = pd.DataFrame({
            "x": [xmin, xmax, xmax, xmin, xmin],
            "y": [ymin, ymin, ymax, ymax, ymin]
        })

        return (
            alt.Chart(df)
            .mark_line(
                color="gray",
                strokeDash=[4,4]
            )
            .encode(
                x="x:Q",
                y="y:Q"
            )
        )

    def plot_ellipse(self, points: np.ndarray) -> alt.Chart:
        df = pd.DataFrame(points, columns=["x", "y"]).reset_index()  # adds 'index' column
        return (
            alt.Chart(df)
            .mark_line()
            .encode(
                x=alt.X("x:Q", scale=alt.Scale(zero=False)),
                y=alt.Y("y:Q", scale=alt.Scale(zero=False)),
                order=alt.Order("index:Q")  # now 'index' exists and is quantitative
            )
        )
        
    def plot(self, obj: Ellipse) -> alt.TopLevelMixin:
        ellipse = obj;
        points = obj.sample(num_points=200);
        
        return (
            self.plot_axes(ellipse)
            + self.plot_ellipse(points)
            + self.plot_major_axis(ellipse)
            + self.plot_minor_axis(ellipse)
            + self.plot_center(ellipse)
            + self.plot_vertices(ellipse)
            + self.plot_covertices(ellipse)
            + self.plot_foci(ellipse)
            
        );

In [ ]:
e: Ellipse = Ellipse(a=-12, b=6, center=(0, 0), rotation=np.pi/2);
f: Ellipse = Ellipse(a=-12, b=6, center=(0, 0), rotation=0);
plotter: EllipsePlotter = EllipsePlotter();
pl_1 = plotter.plot(e);
pl_2 = plotter.plot(f);

pl = pl_1 + pl_2
pl.show()

alt.LayerChart(...)